# Training Workbench

This notebook is the supported interactive entrypoint for training and Optuna tuning inside the Jupyter container. It reuses the callable building blocks from `train.py`, so MLflow logging, local checkpointing, MLflow artifact logging for the best checkpoint, MLflow Model Registry registration of the best model, and study orchestration stay aligned with the script runtime.


In [ ]:
from __future__ import annotations

import copy
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import torch

def resolve_app_root() -> Path:
    candidates: list[Path] = []
    configured_root = os.environ.get("APP_ROOT")
    if configured_root:
        candidates.append(Path(configured_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, cwd.parent, Path("/app")])

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "train.py").exists() and (candidate / "config.yaml").exists():
            return candidate

    raise FileNotFoundError(
        f"Unable to resolve the training app root from: {[str(path) for path in candidates]}"
    )


APP_ROOT = resolve_app_root()
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import train as train_module

from train import (
    BEST_CHECKPOINT_ARTIFACT_PATH,
    TrainingContext,
    TrainingResult,
    deserialize_training_result,
    ensure_supported_tune_runtime,
    find_git_repo_root,
    get_optuna_study_name,
    load_config,
    prepare_model_cache,
    resolve_accelerate_config_path,
    resolve_hf_cache_dir,
    resolve_model_source,
    resolve_study_output_dir,
    resolve_tune_num_processes,
    run_optuna_study,
    sanitize_study_name,
    serialize_training_context,
    set_nested_value,
    summarize_training_config,
    train_worker,
    write_json_file,
    write_yaml_file,
)

CONFIG_PATH = APP_ROOT / "config.yaml"
ACCELERATE_CONFIG_PATH = Path(resolve_accelerate_config_path()).expanduser()
if not ACCELERATE_CONFIG_PATH.is_absolute():
    ACCELERATE_CONFIG_PATH = (APP_ROOT / ACCELERATE_CONFIG_PATH).resolve()
TRAIN_SCRIPT_PATH = Path(train_module.__file__).resolve()
LAST_RESULT: TrainingResult | None = None
LAST_STUDY = None


## Parameters

Edit this cell before running the execution cells below.


In [ ]:
MODE = "train"
EXECUTION_MODE = "interactive"
EXPERIMENT_NAME_OVERRIDE = None
EXACT_RUNTIME_NUM_PROCESSES = None
PREPARE_MODEL_CACHE = False

CONFIG_OVERRIDES = {
    # "data.source": "mock",
    # "training.num_epochs": 1,
    # "training.per_device_train_batch_size": 4,
}


In [ ]:
def build_runtime_config() -> dict:
    cfg = copy.deepcopy(load_config(str(CONFIG_PATH)))
    for dotted_path, value in CONFIG_OVERRIDES.items():
        set_nested_value(cfg, dotted_path, value)

    if EXPERIMENT_NAME_OVERRIDE:
        cfg.setdefault("mlflow", {})
        cfg["mlflow"]["active_experiment_name"] = EXPERIMENT_NAME_OVERRIDE

    cfg.setdefault("optuna", {})
    cfg["optuna"]["study_name"] = get_optuna_study_name(cfg)
    return cfg


def resolve_exact_runtime_num_processes() -> int:
    if EXACT_RUNTIME_NUM_PROCESSES is None:
        return resolve_tune_num_processes()

    parsed = int(EXACT_RUNTIME_NUM_PROCESSES)
    if parsed < 1:
        raise ValueError(f"EXACT_RUNTIME_NUM_PROCESSES must be at least 1, got {parsed}.")
    return parsed


def resolve_registered_model_name(cfg: dict) -> str | None:
    if not cfg.get("model_registry", {}).get("log_to_mlflow_model_registry", True):
        return None
    return cfg.get("model_registry", {}).get("model_name")


def resolve_best_checkpoint_artifact_path(run_id: str | None) -> str | None:
    if not run_id:
        return None
    return BEST_CHECKPOINT_ARTIFACT_PATH


def resolve_optuna_search_space_artifact_path(cfg: dict) -> str:
    experiment_name = cfg.get("mlflow", {}).get("active_experiment_name") or get_optuna_study_name(cfg)
    return f"Optuna/{sanitize_study_name(experiment_name)}.yaml"


def print_section(title: str, payload) -> None:
    print(f"\n=== {title} ===")
    if isinstance(payload, (dict, list)):
        print(json.dumps(payload, indent=2, sort_keys=True, default=str))
    else:
        print(payload)


def run_interactive_training(cfg: dict) -> TrainingResult:
    context = TrainingContext(mode="train")
    return train_worker(copy.deepcopy(cfg), context=context)


def run_exact_runtime_training(cfg: dict) -> TrainingResult:
    exact_runtime_num_processes = resolve_exact_runtime_num_processes()
    with tempfile.TemporaryDirectory(prefix="notebook-train-") as temp_dir:
        temp_root = Path(temp_dir)
        config_path = temp_root / "resolved_config.yaml"
        context_path = temp_root / "training_context.json"
        result_path = temp_root / "result.json"
        progress_path = temp_root / "progress.jsonl"

        context = TrainingContext(
            mode="train",
            progress_file=str(progress_path),
            result_file=str(result_path),
        )
        write_yaml_file(config_path, cfg)
        write_json_file(context_path, serialize_training_context(context))

        command = [
            sys.executable,
            "-m",
            "accelerate.commands.launch",
            "--num_processes",
            str(exact_runtime_num_processes),
            "--config_file",
            str(ACCELERATE_CONFIG_PATH),
            str(TRAIN_SCRIPT_PATH),
            "--config",
            str(config_path),
            "--mode",
            "train",
            "--context-file",
            str(context_path),
        ]
        print_section("Exact-Runtime Command", " ".join(command))
        completed = subprocess.run(command, check=False)

        if not result_path.exists():
            raise RuntimeError(
                f"Exact-runtime training exited with code {completed.returncode} before writing {result_path}."
            )

        payload = json.loads(result_path.read_text(encoding="utf-8"))
        status = payload.get("status")
        if status == "complete":
            return deserialize_training_result(payload["result"])
        if status == "pruned":
            raise RuntimeError(payload.get("message", "Training run was pruned."))

        raise RuntimeError(
            f"Exact-runtime training failed with status={status!r}: "
            f"{payload.get('error_type', 'RuntimeError')}: {payload.get('error_message', 'unknown error')}"
        )


def summarize_execution_plan(cfg: dict) -> None:
    print_section("Execution Parameters", {
        "mode": MODE,
        "execution_mode": EXECUTION_MODE,
        "experiment_name_override": EXPERIMENT_NAME_OVERRIDE,
        "exact_runtime_num_processes": resolve_exact_runtime_num_processes(),
        "config_overrides": CONFIG_OVERRIDES,
    })
    print_section("Training Summary", summarize_training_config(cfg))
    print_section("Evaluation", cfg["evaluation"])
    print_section("Model Registry", cfg.get("model_registry", {}))
    if MODE == "tune":
        print_section("Optuna", cfg.get("optuna", {}))
    print_section("Resolved Paths", {
        "config_path": str(CONFIG_PATH),
        "train_script": str(TRAIN_SCRIPT_PATH),
        "accelerate_config_path": str(ACCELERATE_CONFIG_PATH),
        "checkpoint_dir": cfg["checkpointing"]["checkpoint_dir"],
        "hf_cache_dir": str(resolve_hf_cache_dir(cfg)),
        "model_source": resolve_model_source(cfg),
        "study_output_dir": str(resolve_study_output_dir(cfg)),
    })


## Bootstrap Checks

Run this once after the kernel starts. It verifies the repository wiring expected by the notebook and by `train.py`.


In [ ]:
def resolve_git_commit(repo_root: Path | None) -> str:
    if repo_root is None:
        return "unavailable"

    git_result = subprocess.run(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"],
        check=False,
        capture_output=True,
        text=True,
    )
    if git_result.returncode == 0:
        return git_result.stdout.strip() or "unavailable"

    git_dir = repo_root / ".git"
    head_path = git_dir / "HEAD"
    if not head_path.exists():
        return "unavailable"

    head_value = head_path.read_text(encoding="utf-8").strip()
    if not head_value:
        return "unavailable"
    if not head_value.startswith("ref: "):
        return head_value

    ref_name = head_value[5:].strip()
    ref_path = git_dir / ref_name
    if ref_path.exists():
        return ref_path.read_text(encoding="utf-8").strip() or "unavailable"

    packed_refs_path = git_dir / "packed-refs"
    if packed_refs_path.exists():
        for raw_line in packed_refs_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or line.startswith("^"):
                continue
            sha, _, packed_ref = line.partition(" ")
            if packed_ref.strip() == ref_name:
                return sha.strip() or "unavailable"

    return "unavailable"


repo_root = find_git_repo_root(APP_ROOT)
required_env = [
    "MLFLOW_TRACKING_URI",
    "NUM_PROCESSES",
]

git_commit = resolve_git_commit(repo_root)

bootstrap_status = {
    "cwd": str(Path.cwd()),
    "python_executable": sys.executable,
    "config_exists": CONFIG_PATH.exists(),
    "train_script_exists": TRAIN_SCRIPT_PATH.exists(),
    "accelerate_config_exists": ACCELERATE_CONFIG_PATH.exists(),
    "git_repo_root": str(repo_root) if repo_root else None,
    "git_commit": git_commit,
    "gpu_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
    "cuda_version": torch.version.cuda,
    "num_processes_env": os.environ.get("NUM_PROCESSES"),
    "missing_env": [key for key in required_env if not os.environ.get(key)],
}
print_section("Bootstrap Status", bootstrap_status)

assert CONFIG_PATH.exists(), f"Missing config file: {CONFIG_PATH}"
assert TRAIN_SCRIPT_PATH.exists(), f"Missing training script: {TRAIN_SCRIPT_PATH}"
assert repo_root is not None, f"Expected a git repo containing {APP_ROOT}."


## Build Runtime Config

This cell loads the repo `config.yaml`, applies notebook overrides, and shows the effective settings before you train or tune.


In [ ]:
cfg = build_runtime_config()
summarize_execution_plan(cfg)

if PREPARE_MODEL_CACHE:
    cached_model_path = prepare_model_cache(cfg)
    print_section("Prepared Model Cache", cached_model_path)
else:
    print("Set PREPARE_MODEL_CACHE = True if you want to warm the Hugging Face cache before execution.")


## Interactive Training

Use this when you want to stay in-kernel for fast iteration on a single controller process while still using the real `train_worker` implementation.


In [ ]:
LAST_RESULT = None
LAST_STUDY = None

if MODE != "train" or EXECUTION_MODE != "interactive":
    print("Skip this cell unless MODE='train' and EXECUTION_MODE='interactive'.")
else:
    interactive_cfg = build_runtime_config()
    LAST_RESULT = run_interactive_training(interactive_cfg)
    print_section("Interactive Training Result", {
        "run_id": LAST_RESULT.run_id,
        "best_metric_name": LAST_RESULT.best_metric_name,
        "best_metric": LAST_RESULT.best_metric,
        "best_checkpoint": LAST_RESULT.best_checkpoint,
        "final_metrics": LAST_RESULT.final_metrics,
    })


## Exact-Runtime Training

Use this when you want the notebook to launch the same Accelerate CLI path that the script uses in containerized training.


In [ ]:
LAST_RESULT = None
LAST_STUDY = None

if MODE != "train" or EXECUTION_MODE != "exact-runtime":
    print("Skip this cell unless MODE='train' and EXECUTION_MODE='exact-runtime'.")
else:
    exact_runtime_cfg = build_runtime_config()
    LAST_RESULT = run_exact_runtime_training(exact_runtime_cfg)
    print_section("Exact-Runtime Training Result", {
        "run_id": LAST_RESULT.run_id,
        "best_metric_name": LAST_RESULT.best_metric_name,
        "best_metric": LAST_RESULT.best_metric,
        "best_checkpoint": LAST_RESULT.best_checkpoint,
        "final_metrics": LAST_RESULT.final_metrics,
    })


## Tune With Optuna

Tune mode already uses subprocess-launched Accelerate trials internally, so this cell is the exact study controller path from `train.py`.


In [ ]:
LAST_RESULT = None
LAST_STUDY = None

if MODE != "tune":
    print("Skip this cell unless MODE='tune'.")
else:
    tune_cfg = build_runtime_config()
    ensure_supported_tune_runtime()
    LAST_STUDY = run_optuna_study(copy.deepcopy(tune_cfg))
    print_section("Tune Result", {
        "study_name": LAST_STUDY.study_name,
        "best_value": LAST_STUDY.best_value,
        "best_trial_number": LAST_STUDY.best_trial.number,
        "best_metric_name": LAST_STUDY.best_trial.user_attrs.get("best_metric_name"),
        "best_run_id": LAST_STUDY.best_trial.user_attrs.get("run_id"),
        "best_checkpoint": LAST_STUDY.best_trial.user_attrs.get("best_checkpoint"),
        "best_params": LAST_STUDY.best_trial.params,
        "study_output_dir": LAST_STUDY.user_attrs.get("study_output_dir", str(resolve_study_output_dir(tune_cfg))),
        "best_config_path": LAST_STUDY.user_attrs.get("best_config_path"),
        "trial_summary_path": LAST_STUDY.user_attrs.get("trial_summary_path"),
    })


## Result Summary

Re-run this after a training or tuning cell to inspect the latest outputs, the current MLflow artifact paths, and the registered model name for the best model when registry logging is enabled.


In [ ]:
if LAST_RESULT is not None:
    summary_cfg = copy.deepcopy(LAST_RESULT.resolved_config)
    print_section("Last Training Run", {
        "run_id": LAST_RESULT.run_id,
        "best_metric_name": LAST_RESULT.best_metric_name,
        "best_metric": LAST_RESULT.best_metric,
        "best_checkpoint": LAST_RESULT.best_checkpoint,
        "best_checkpoint_artifact_path": resolve_best_checkpoint_artifact_path(LAST_RESULT.run_id),
        "mlflow_registered_model_name": resolve_registered_model_name(summary_cfg),
        "checkpoint_dir": summary_cfg["checkpointing"]["checkpoint_dir"],
        "model_registry": summary_cfg.get("model_registry", {}),
        "final_metrics": LAST_RESULT.final_metrics,
    })
elif LAST_STUDY is not None:
    best_trial_attrs = dict(LAST_STUDY.best_trial.user_attrs)
    summary_cfg = copy.deepcopy(best_trial_attrs.get("resolved_config") or build_runtime_config())
    print_section("Last Study", {
        "study_name": LAST_STUDY.study_name,
        "best_value": LAST_STUDY.best_value,
        "best_trial_number": LAST_STUDY.best_trial.number,
        "best_metric_name": best_trial_attrs.get("best_metric_name"),
        "best_run_id": best_trial_attrs.get("run_id"),
        "best_checkpoint": best_trial_attrs.get("best_checkpoint"),
        "best_params": LAST_STUDY.best_trial.params,
        "study_output_dir": LAST_STUDY.user_attrs.get("study_output_dir", str(resolve_study_output_dir(summary_cfg))),
        "best_config_path": LAST_STUDY.user_attrs.get("best_config_path", str(resolve_study_output_dir(summary_cfg) / "best_config.yaml")),
        "trial_summary_path": LAST_STUDY.user_attrs.get("trial_summary_path", str(resolve_study_output_dir(summary_cfg) / "trial_summary.json")),
        "optuna_search_space_artifact_path": resolve_optuna_search_space_artifact_path(summary_cfg),
        "mlflow_registered_model_name": resolve_registered_model_name(summary_cfg),
        "model_registry": summary_cfg.get("model_registry", {}),
        "final_metrics": best_trial_attrs.get("final_metrics", {}),
    })
else:
    print("No training or tuning result has been captured in this kernel yet.")
